In [1]:
import pandas as pd
import altair as alt

# Disable max rows limit (needed for large datasets)
alt.data_transformers.disable_max_rows()

# Load the dataset directly from URL
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
df = pd.read_csv(url)

# Quick look at the data
print(df.shape)
df.head()

(4747, 29)


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [2]:
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']
observed               object
location_details       object
county                 object
state                  object
season                 object
title                  object
latitude              float64
longitude             float64
date                   object
number                float64
classification         object
geohash                object
temperature_high      float64
temperature_mid       float64
temperature_low       float64
dew_point             float64
humidity              float64
cloud_cover           float64
moon_phase            float64
precip_intensity      float

In [3]:
# Count sightings per state and take top 20
state_counts = (
    df['state']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'state', 'state': 'count'})
    .head(20)
)

# Rename columns clearly
state_counts.columns = ['state', 'count']

plot1 = alt.Chart(state_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Sightings'),
    y=alt.Y('state:N', sort='-x', title='State'),
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='teals'),
        title='Sightings'
    ),
    tooltip=['state:N', 'count:Q']
).properties(
    title='Top 20 States by Bigfoot Sightings',
    width=500,
    height=400
)

plot1

alt.Chart(...)

In [12]:
us_states_url = "https://cdn.jsdelivr.net/npm/vega-datasets@1/data/us-10m.json"

# Prepare per-classification counts
state_class = df.groupby(['state', 'classification']).size().reset_index(name='count')
state_class['id'] = state_class['state'].map(state_id_map)
state_class = state_class.dropna(subset=['id'])
state_class['id'] = state_class['id'].astype(int)

# Check what classifications exist
print(state_class['classification'].unique())
print(state_class.head(10))

['Class A' 'Class B' 'Class C']
      state classification  count  id
0   Alabama        Class A     58   1
1   Alabama        Class B     32   1
2   Alabama        Class C      1   1
3    Alaska        Class A     12   2
4    Alaska        Class B      8   2
5   Arizona        Class A     27   4
6   Arizona        Class B     57   4
7  Arkansas        Class A     61   5
8  Arkansas        Class B     31   5
9  Arkansas        Class C      3   5


In [20]:
us_states_url = "https://cdn.jsdelivr.net/npm/vega-datasets@1/data/us-10m.json"
us_topo = alt.topo_feature(us_states_url, 'states')

# Separate dataframes for each class
state_classA = state_class[state_class['classification'] == 'Class A'][['id', 'state', 'count']].rename(columns={'count': 'Class A'})
state_classB = state_class[state_class['classification'] == 'Class B'][['id', 'state', 'count']].rename(columns={'count': 'Class B'})
state_classC = state_class[state_class['classification'] == 'Class C'][['id', 'state', 'count']].rename(columns={'count': 'Class C'})

# Merge all into one dataframe
state_merged = state_classA.merge(state_classB, on=['id', 'state'], how='outer').merge(state_classC, on=['id', 'state'], how='outer').fillna(0)
state_merged[['Class A', 'Class B', 'Class C']] = state_merged[['Class A', 'Class B', 'Class C']].astype(int)

# Dropdown
class_dropdown = alt.binding_select(
    options=['Class A', 'Class B', 'Class C'],
    name='Select Classification: '
)
class_param = alt.param(
    name='selected_class',
    bind=class_dropdown,
    value='Class A'
)

background = alt.Chart(us_topo).mark_geoshape(
    fill='#d3d3d3',   # slightly darker gray so it contrasts better with light blue
    stroke='white'
).project('albersUsa').properties(
    width=700,
    height=450
)

choropleth = alt.Chart(us_topo).mark_geoshape(
    stroke='white'
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=state_merged,
        key='id',
        fields=['state', 'Class A', 'Class B', 'Class C']
    )
).transform_calculate(
    selected_count="selected_class == 'Class A' ? datum['Class A'] : selected_class == 'Class B' ? datum['Class B'] : datum['Class C']"
).encode(
    color=alt.condition(
    'datum.selected_count > 0',
    alt.Color(
        'selected_count:Q',
        scale=alt.Scale(
            scheme='blues',
            domainMin=1
        ),
        title='Number of Sightings'
    ),
    alt.value('lightgray')
),
    tooltip=[
        alt.Tooltip('state:N', title='State'),
        alt.Tooltip('selected_count:Q', title='Sightings')
    ]
).add_params(
    class_param
).project('albersUsa')

plot2 = (background + choropleth).properties(
    title=alt.TitleParams(
        text='Bigfoot Sightings by State and Classification',
        subtitle='Use the dropdown below to filter by Class A, B, or C sightings'
    )
)

plot2

alt.LayerChart(...)

In [21]:
plot1.save('plot1.json')
plot2.save('plot2.json')